# AgriRAG Naija: A Retrieval-Augmented Generation Assistant for Nigerian Smallholder Farmers

**Author:** Samuel Yaula Dutse (SamDutse) — Lead Data Scientist, Bluehouse Technologies Ltd. | AI/Data Science Instructor, Nexus Hub Limited

**Inspired by:** an introductory RAG lesson that built a Berlin-facts tour-guide chatbot. This notebook keeps the same architecture and teaches the same core ideas, but rebuilds it around a real problem in Nigerian agriculture.

---

## The problem we're solving

Nigeria has roughly **1 agricultural extension worker for every 3,000+ farmers** (the recommended ratio is closer to 1:500–1:800). Most smallholder farmers — the people growing the maize, cassava, rice, and vegetables that feed the country — have no reliable, on-demand way to ask basic agronomic questions:

- *"When should I plant maize in Kaduna State?"*
- *"My cassava leaves have yellow patterns, what is wrong?"*
- *"How do I store maize to stop it from getting moldy?"*

Generic chatbots either don't know Nigerian-specific agronomy (planting calendars differ hugely from the US/Europe), or they hallucinate confident-sounding but wrong advice — which is dangerous when it affects a family's only source of income.

**Our approach:** build a small RAG (Retrieval-Augmented Generation) system that only answers from a curated set of verified agricultural facts. If the answer isn't in the knowledge base, the assistant says so instead of guessing. This is the same "grounded answers over guessed answers" principle used in the original Berlin tour-guide notebook — just applied to a problem that actually matters for livelihoods.

This notebook is meant to be:
1. A **teaching walkthrough** — read top to bottom to learn how RAG works, piece by piece.
2. A **portfolio project** — clone it, extend it, push it to GitHub.
3. A **starting point for students** — every section ends with a short exercise.


## What is RAG, in one paragraph?

A Large Language Model (LLM) only "knows" what was in its training data, and it has no idea about your private/specialized knowledge base — in our case, curated Nigerian agronomy facts. **Retrieval-Augmented Generation** fixes this in two steps:

1. **Retrieve**: given a question, search a knowledge base for the most relevant pieces of text (using vector similarity, not keyword matching).
2. **Generate**: hand those retrieved pieces of text to an LLM as *context*, and instruct it to answer using only that context.

The result is an assistant that is both fluent (thanks to the LLM) and *grounded* (thanks to retrieval) — it can't easily make things up because it's told exactly what facts it's allowed to use.


# 1. Setup

In [ ]:
# If you're running this in Google Colab, store your Hugging Face token as a Colab secret named HF_TOKEN.
# If you're running this locally / outside Colab, it will fall back to a manual prompt.
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_TOKEN')
except ModuleNotFoundError:
    import getpass
    hf_key = getpass.getpass("Enter your Hugging Face token: ")


In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = hf_key


In [ ]:
!pip install langchain-huggingface sentence-transformers langchain_community faiss-cpu -q


In [ ]:
# Import the libraries
from langchain.docstore.document import Document
from langchain.vectorstores.faiss import FAISS
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from IPython.display import display, Markdown


# 2. The Knowledge Base

Every RAG system starts with the same question: **what should the model be allowed to know?**

For a real deployment, this would come from verified sources — IITA (International Institute of Tropical Agriculture) advisories, NAERLS (National Agricultural Extension and Research Liaison Services) bulletins, ADP (Agricultural Development Programme) extension manuals, or FMARD (Federal Ministry of Agriculture and Rural Development) guidance.

For this teaching notebook, we use a **synthetic but agronomically realistic** set of facts covering staple crops, common pests/diseases, livestock, storage, and government support programs relevant to Nigerian smallholder farmers. Each fact is a short, self-contained sentence — this makes retrieval more precise, since we're not chunking large documents (that's a good next step for students to try, see the exercises).

> ⚠️ **Note:** these facts were written for this teaching exercise and simplified for clarity. Before using anything like this in production, every fact should be verified against a real agronomic source.


In [ ]:
# Synthetic knowledge base: 62 facts about Nigerian smallholder agriculture
documents = [
    # --- Maize ---
    "Maize is Nigeria's most widely cultivated cereal crop, grown across almost all agroecological zones.",
    "In Northern Nigeria, maize is typically planted at the onset of the rains, around May to June.",
    "In Southern Nigeria, maize can be planted twice a year: an early crop around March and a late crop around August.",
    "Fall armyworm is one of the most destructive pests of maize in Nigeria and can wipe out an entire field if untreated.",
    "Fall armyworm damage on maize typically appears as ragged holes in young leaves and sawdust-like frass in the whorl.",
    "Farmers can scout for fall armyworm by checking maize whorls weekly during the first six weeks after planting.",
    "Maize streak virus, spread by leafhoppers, causes yellow streaking on leaves and stunted growth.",
    "Recommended maize planting spacing in Nigeria is commonly 75cm between rows and 25cm between plants.",
    "NPK 15:15:15 fertilizer is commonly applied to maize about two to three weeks after planting.",
    "A second application of urea fertilizer is often recommended for maize around five to six weeks after planting.",

    # --- Cassava ---
    "Cassava is a staple root crop in Nigeria and is highly tolerant of poor soils and erratic rainfall.",
    "Cassava can be planted almost any time of year in the humid south, but early rains give the best establishment.",
    "Cassava mosaic disease causes yellow mottling and leaf distortion and spreads through infected cuttings and whiteflies.",
    "Farmers are advised to source cassava stem cuttings from disease-free, certified plants to avoid mosaic disease.",
    "Cassava is typically ready for harvest between 8 and 18 months after planting, depending on the variety.",
    "Nigeria is the world's largest producer of cassava, though much of it is consumed domestically rather than exported.",
    "Garri, fufu, and cassava flour are among the most common processed cassava products in Nigeria.",

    # --- Rice ---
    "Rice cultivation in Nigeria includes upland, lowland rainfed, and irrigated systems, each with different water needs.",
    "Transplanting rice seedlings at 21 days old generally gives better yields than direct seeding in lowland systems.",
    "Rice blast disease appears as diamond-shaped lesions on leaves and can be worsened by excessive nitrogen fertilizer.",
    "The Anchor Borrowers' Programme has supported many Nigerian rice farmers with input financing and off-take arrangements.",
    "Proper water management, including alternate wetting and drying, can reduce water use in irrigated rice without cutting yield.",

    # --- Yam ---
    "Nigeria produces more yam than any other country in the world, concentrated mainly in the Middle Belt and South.",
    "Yam is usually planted between February and April, ahead of the main rains.",
    "Staking yam vines helps them climb, improves sunlight exposure, and can increase tuber yield.",
    "Yam anthracnose disease causes dark leaf spots and can significantly reduce vine vigor if untreated.",

    # --- Cowpea / beans ---
    "Cowpea, commonly called beans in Nigeria, is an important source of dietary protein and fixes nitrogen in the soil.",
    "Cowpea is often intercropped with cereals like maize or sorghum to make efficient use of farmland.",
    "The pod-sucking bug and cowpea aphid are common pests that reduce cowpea yield and seed quality.",
    "Early maturing cowpea varieties can be ready for harvest in as little as 60 to 70 days.",

    # --- Tomato and vegetables ---
    "Tuta absoluta, sometimes called the tomato leafminer, has caused severe tomato losses across Northern Nigeria in recent years.",
    "Tomato Ebola was a popular local name given to a major Tuta absoluta outbreak that devastated tomato farms in 2016.",
    "Staking and pruning tomato plants improves air circulation and reduces the spread of fungal diseases.",
    "Pepper and tomato seedlings are usually raised in a nursery for about three to four weeks before transplanting.",

    # --- Soil and fertilizer ---
    "Soil testing before planting helps farmers apply the right type and amount of fertilizer instead of guessing.",
    "The Growth Enhancement Support Scheme introduced electronic vouchers to help Nigerian farmers access subsidized fertilizer.",
    "Organic manure such as poultry droppings or compost can improve soil structure alongside inorganic fertilizer.",
    "Continuous cropping without fallow periods or organic matter can gradually deplete soil fertility.",
    "Liming acidic soils can improve nutrient availability, particularly in parts of the humid forest zone.",

    # --- Irrigation and climate ---
    "Nigeria has three broad agroecological patterns: the arid/semi-arid north, the Guinea savanna middle belt, and the humid south.",
    "The rainy season in Northern Nigeria is shorter and less predictable than in the south, making irrigation valuable there.",
    "Fadama irrigation schemes support dry-season farming along river floodplains, especially for vegetables and rice.",
    "Climate variability has shifted the onset of rains in parts of Nigeria, complicating traditional planting calendars.",

    # --- Poultry and livestock ---
    "Newcastle disease is one of the most economically damaging viral diseases affecting local and commercial poultry in Nigeria.",
    "Routine vaccination against Newcastle disease is recommended for backyard and commercial poultry flocks.",
    "Coccidiosis in poultry is caused by parasites and commonly affects young chicks in humid, poorly drained pens.",
    "Fulani pastoralists practice transhumance, moving cattle seasonally in search of pasture and water.",
    "Foot-and-mouth disease can spread rapidly among cattle herds and is a major concern for Nigerian livestock farmers.",
    "Proper deworming schedules for goats and sheep help reduce losses from internal parasites.",

    # --- Post-harvest and storage ---
    "Aflatoxin contamination in poorly dried maize and groundnuts can pose serious health risks if consumed.",
    "Sun-drying maize to below 13% moisture content before storage helps prevent mold growth.",
    "Hermetic storage bags, often called PICS bags, can protect stored grain from insect pests without chemical treatment.",
    "Post-harvest losses in Nigeria are estimated to affect a significant share of grain and perishable produce every year.",

    # --- Markets, finance, and extension ---
    "Cooperative societies help smallholder farmers pool resources for input purchases and access to credit.",
    "NIRSAL, the Nigeria Incentive-Based Risk Sharing System for Agricultural Lending, was created to de-risk agricultural lending.",
    "Mobile money and agent banking have made it easier for rural farmers to receive payments and government support.",
    "Local market days in many rural communities determine when farmers sell produce and buy inputs.",
    "Agricultural extension agents in Nigeria are often responsible for far more farmers than international best-practice ratios recommend.",
    "Farmer field schools bring extension knowledge directly to communities through hands-on demonstration plots.",
    "Weather advisory services delivered via SMS or radio help farmers time planting and spraying decisions.",
    "Contract farming arrangements can guarantee farmers a buyer, but often require them to follow specific production standards.",
    "Warehouse receipt systems allow farmers to store grain and borrow against it instead of selling immediately at low harvest-time prices.",
]


In [ ]:
# Check how many documents we have
print(f"We have {len(documents)} documents")


### 🧪 Exercise 1
Add 5–10 more facts of your own — maybe about a crop or livestock topic relevant to your own state or region. Keep each fact to one clear sentence, the same way the examples above are written.


# 3. Embeddings — Turning Text Into Vectors

An embedding model converts each sentence into a vector of numbers such that sentences with similar *meaning* end up close together in that vector space — even if they don't share the same words. This is what lets retrieval work on meaning, not just keyword overlap.

We use `sentence-transformers/all-MiniLM-L6-v2`, a small, fast, widely-used embedding model — good enough for a teaching notebook and light enough to run on a free Colab CPU.


In [ ]:
# Wrap each string in a Document object (LangChain's standard text container)
docs = [Document(page_content=text) for text in documents]


In [ ]:
# Use a transformer-based embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
# Create a FAISS vector store from the documents
# FAISS (Facebook AI Similarity Search) indexes the vectors so we can search them quickly
faiss_store = FAISS.from_documents(docs, embedding_model)


In [ ]:
index = faiss_store.index

# Print total number of indexed vectors
print(f"Total number of indexed vectors: {index.ntotal}")

# Print total number of dimensions per vector
print(f"Total number of dimensions: {index.d}")

# Print the embedding for the first document, just to see what one actually looks like
print(f"Embedding for the first document:\n{index.reconstruct(0)}")


# 4. Retrieval System — Finding the Right Facts

Given a farmer's question, we embed the question the same way we embedded the documents, then find the `k` documents whose vectors are closest to it. This is a **similarity search**, not a keyword search — so "yellow spots on my cassava leaves" can still retrieve the cassava mosaic disease fact even without matching words exactly.


In [ ]:
# Try a raw similarity search
query = "What can I do about pests on my maize?"
k = 5
retrieved_docs = faiss_store.similarity_search(query, k)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"{i}. {doc.page_content}")


In [ ]:
# Wrap retrieval in a reusable function
def get_relevant_documents(query, k=5):
    return faiss_store.similarity_search(query, k)


### 🧪 Exercise 2
Try `faiss_store.similarity_search_with_score(query, k)` instead of `similarity_search`. What do the scores tell you? Try a `k` of 2 vs. 10 — how does the quality of the retrieved context change?


# 5. Generative System — Turning Retrieved Facts Into an Answer

Now we hand the retrieved facts to an LLM as context, along with a system prompt that constrains it to only use that context. We give the assistant a persona — a friendly Nigerian agricultural extension officer — since tone matters when the audience is a farmer, not a developer.

We use `microsoft/Phi-3.5-mini-instruct` via the Hugging Face Inference Endpoint, the same lightweight instruction-tuned model used in the original lesson.


In [ ]:
# Load the LLM
llm = HuggingFaceEndpoint(
    repo_id="microsoft/Phi-3.5-mini-instruct",
    task="text-generation"
)
chat_model = ChatHuggingFace(llm=llm)


In [ ]:
# Define the system and human messages
def generative_system(query, context):
    messages = [
        SystemMessage(content=f"""
        You are a friendly Nigerian agricultural extension officer helping smallholder farmers.
        Explain things in simple, clear English a farmer with no formal agronomy training can follow.
        Only answer using information from {context}.
        If the context does not contain the answer, say you don't have that information yet
        and suggest the farmer contact their local extension office."""),
        HumanMessage(content=f"Answer this farmer's question: {query}, based on this context: {context}")
    ]
    ai_output = chat_model.invoke(messages)
    return display(Markdown(ai_output.content))


### 🧪 Exercise 3
Try rewriting the system prompt so the assistant replies in **Naija Pidgin** instead of standard English. This is a great extension if you're already exploring Pidgin fine-tuning — see how far prompting alone gets you before you'd need a fine-tuned model.


# 6. Combining Retrieval and Generation = RAG

This is the whole system in five lines: retrieve the most relevant facts, then generate a grounded answer from them.


In [ ]:
# Build the RAG system
def rag(query):
    context = get_relevant_documents(query)
    return generative_system(query, context)


In [ ]:
# Test the RAG system with a question our knowledge base can answer
query = "My maize leaves have small holes and I see sawdust-like material in the middle. What is happening?"
rag(query)


In [ ]:
# Prepare test queries, including one the knowledge base genuinely cannot answer
query_list = [
    "When should I plant maize in Northern Nigeria?",
    "How can I stop my stored maize from getting moldy?",
    "What is the best way to grow avocados in Nigeria?",  # not in our knowledge base
]


In [ ]:
# Test the RAG system across all queries
for query in query_list:
    print(f"Q: {query}")
    rag(query)
    print("\n---\n")


Notice the last question — about avocados — isn't covered in our knowledge base. A well-grounded RAG system should say it doesn't have that information rather than inventing a confident-sounding answer. If your assistant hallucinates here, that's a signal to tighten the system prompt or add a lower similarity-score threshold before passing context to the LLM.


# 7. What's Actually Happening Under the Hood

```
Farmer's question
       │
       ▼
 Embed the question  ──────────────►  vector
       │
       ▼
 Search FAISS index for nearest k document vectors
       │
       ▼
 Retrieved facts (context)
       │
       ▼
 System prompt + context + question  ──────────►  LLM (Phi-3.5-mini)
       │
       ▼
 Grounded answer in a farmer-friendly persona
```

Every RAG system, no matter how advanced, is a variation on this same loop: **embed → retrieve → augment the prompt → generate**.


# 8. Exercises to Extend This Project

Pick one or more of these to make the project your own before pushing it to GitHub:

1. **Chunking real documents.** Replace the synthetic fact list with real extension PDFs (e.g. IITA or NAERLS bulletins), split into chunks using `RecursiveCharacterTextSplitter`, and re-run the pipeline.
2. **Source citation.** Modify `generative_system` to also display which retrieved facts were used, so farmers (or reviewers) can verify the answer.
3. **Similarity score threshold.** Use `similarity_search_with_score` and discard retrieved documents below a relevance threshold, instead of always returning `k` documents.
4. **Multilingual/Pidgin support.** Extend the system prompt (or plug in a fine-tuned model) to answer in Naija Pidgin or Hausa.
5. **Evaluation.** Write 15–20 test questions with known correct answers, and manually score how often the RAG system is accurate vs. how often it says "I don't know."
6. **Deployment.** Wrap `rag()` in a Gradio interface and deploy it to Hugging Face Spaces, following the same workflow used for other projects like CropGuard.
7. **Voice input.** For real farmer usability, add speech-to-text (e.g. `faster-whisper`) so farmers can ask questions by voice instead of typing.

## Limitations to be upfront about

- The knowledge base here is **synthetic** and simplified for teaching — it is not a substitute for verified agronomic guidance.
- Small instruction-tuned models can still occasionally drift from the provided context; always test with adversarial or out-of-scope questions.
- A real deployment would need domain-expert review (agronomists/extension officers) before farmers rely on it.

---

*This notebook was built as a teaching and portfolio project by Samuel Yaula Dutse, adapting the retrieval-augmented generation pattern from an introductory RAG lesson to a problem in Nigerian smallholder agriculture.*
